# MeChess 8/8: Stockfish labels for the human-likeness study (CPU notebook)

An adapter around `chessme humanlike-judge`. For a sample of positions of notebook 6's test set TB it judges the human's move and the 16 most probable moves of Maia-3 23M with Stockfish (a pinned node budget, so the numbers are reproducible), and writes the losses (expected points given away). Two samples, from disjoint games (the split used by `humanlike-adapter`): a **training** sample for the adapter's behaviour-matching term and a larger **test** sample, judged with the same engine so that they are comparable.

**Before you run:** add notebook 6's output (the test sets) and notebook 7's output (the scores) as inputs (*Add Input -> Notebook output files*); *Accelerator -> None*, *Internet -> On*; **Save Version -> Save & Run All (Commit)**. Stockfish is an external download (GPL-3.0, from its official releases); nothing is redistributed.
Judging is the cost: about 7 positions per second on a three-process laptop run; four processes here.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
WORKERS = 4
MODEL = "23m"                                           # which Maia-3 size to judge: 5m, 23m or 79m (its scores must be among notebook 7's output)
TRAIN_PER_REGIME = 30000                                # positions per clock regime (the rarer regimes run out first: all of them are used)
TEST_PER_REGIME = 5000
CANDIDATES = 16                                         # the model's most probable moves judged next to the human's (8 leaves its other moves unjudged)
STOCKFISH_URL = "https://github.com/official-stockfish/Stockfish/releases/download/sf_18/stockfish-ubuntu-x86-64-avx2.tar"
OUT = "/kaggle/working/humanlike" if os.path.exists("/kaggle") else "humanlike_local"
INPUT_DIR = ""                                          # notebooks 6 and 7's output folders; empty: search /kaggle/input

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = sorted(glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True), key=lambda f: "/datasets/" not in f)   # prefer the attached repository dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

In [ ]:
root = INPUT_DIR or "/kaggle/input"
TB = next(iter(sorted(glob.glob(root + "/**/testset_TB.jsonl.gz", recursive=True))), None)
SCORES = next(iter(sorted(glob.glob(root + f"/**/scores_TB_{MODEL}.npz", recursive=True))), None)
print("test set:", TB, "| scores:", SCORES)
if not (TB and SCORES):
    raise SystemExit(f"Add notebook 6's output (testset_TB.jsonl.gz) and notebook 7's output (scores_TB_{MODEL}.npz) as inputs.")
os.makedirs(OUT, exist_ok=True)

## Step 1. Stockfish (external, pinned release)

In [ ]:
os.makedirs("/tmp/sf", exist_ok=True)
sh("curl", "-L", "-s", "-o", "/tmp/sf/sf.tar", STOCKFISH_URL)
sh("tar", "-xf", "/tmp/sf/sf.tar", "-C", "/tmp/sf")
ENGINE = next(f for f in glob.glob("/tmp/sf/**/stockfish*", recursive=True) if os.path.isfile(f) and not f.endswith((".tar", ".txt", ".md")) and os.access(f, os.X_OK))
sh("sh", "-c", f"echo quit | {ENGINE} | head -1 | tee {OUT}/stockfish_version.txt")
sh("sha256sum", "/tmp/sf/sf.tar")

## Step 2. The test sample first (smaller, and what every comparison rests on), then the training sample

In [ ]:
sh(*CLI, "humanlike-judge", "--testset", TB, "--scores", SCORES, "--split", "test", "--per-regime", str(TEST_PER_REGIME), "--workers", str(WORKERS),
   "--candidates", str(CANDIDATES), "--engine", ENGINE, "--out", f"{OUT}/judged_test_{MODEL}.npz")

In [ ]:
sh(*CLI, "humanlike-judge", "--testset", TB, "--scores", SCORES, "--split", "trainval", "--per-regime", str(TRAIN_PER_REGIME), "--workers", str(WORKERS),
   "--candidates", str(CANDIDATES), "--engine", ENGINE, "--out", f"{OUT}/judged_trainval_{MODEL}.npz")

## Step 3. The error profile of the frozen model on the test sample (a first look; the full comparison is done on a laptop)

In [ ]:
sh(*CLI, "humanlike-errors", "--testset", TB, "--scores", SCORES, "--judged", f"{OUT}/judged_test_{MODEL}.npz", "--boot", "500", "--out", f"{OUT}/errors_test_{MODEL}.md")